# 00. Introduction and Objectives

#### 1. Introduction

This notebook aims to explore and test the GDELT DOC 2.0 API as a potential data source for collecting financial news related to NVIDIA (NVDA).

Our main objective is to build a structured dataset of NVIDIA-related news articles, including their headlines, URLs, publication dates, sources, and other available metadata.

This exploratory phase will focus on understanding how the API works, identifying its limitations, and developing a clean and reproducible Python workflow for retrieving and processing news articles.

#### 2. Objectives

The main objectives of this notebook are:

- Understand the GDELT DOC 2.0 API and its Article List mode.

- Identify API limitations and recommended usage practices.

- Develop Python functions to send HTTP GET requests and process JSON responses.

- Retrieve a sample of NVIDIA-related financial news articles.

- Convert the collected metadata into a structured Pandas DataFrame.

- Explore selected article URLs and test headline extraction using web scraping techniques.

- Evaluate data accessibility, extraction success rates, and potential issues.

#### 3. Expected Outcome

At the end of this exploration, we aim to obtain:

- A sample CSV dataset containing NVIDIA-related article metadata.

- A second dataset containing the extracted headlines and scraping results.

- A reusable Python collection workflow.

A documented procedure that can be shared with the other team members to ensure consistent data collection across different time periods.

# 01. Understanding GDELT DOC API

### What is GDELT DOC API?

- GDELT DOC API (Global Database of Events, Language, and Tone) is a free API that allows you to search for news articles from media sources worldwide using keywords, specific dates, and other filters.

For our NVIDIA project, it allows us to:
- Search for articles mentioning NVIDIA.
- Retrieve article headlines, URLs, publication dates, sources and languages.
- Filter articles by date and other criteria.
- Export results in JSON or CSV format.

### How does Article List work?

- Article List (mode=ArtList) is a mode of GDELT DOC 2.0 API that returns a list of news articles matching your search query.

- How does it work?

  1. You define your search keyword: NVIDIA.
  2. You specify the date range.
  3. You select ArtList as the output mode.
  4. GDELT searches its indexed news articles.
  5. It returns a list of matching articles in JSON or CSV format.

- Example API request

https://api.gdeltproject.org/api/v2/doc/doc
?query=NVIDIA
&mode=ArtList
&format=json
&maxrecords=50
&timespan=7d

### API endpoint and request structure

- For GDELT DOC 2.0, the main endpoint is:
 
       https://api.gdeltproject.org/api/v2/doc/doc

- After defining the endpoint, we need to add parameters that describe our search.

        ?query=NVIDIA&mode=ArtList&format=json&maxrecords=50

| Parameter | Purpose | Example |
|---|---|---|
| `query` | Keywords to search for | `NVIDIA` |
| `mode` | Type of result requested | `ArtList` |
| `format` | Response format | `json` |
| `maxrecords` | Maximum articles returned | `50` |
| `timespan` | Relative search period | `7d` |
| `startdatetime` | Beginning of date range | `20261101000000` |
| `enddatetime` | End of date range | `20261102000000` |
| `sort` | Result ordering | `DateDesc` |

- You don't need to use all these parameters in every request.

### API limitations and usage rules

##### 1.1 Historical search period

- GDELT DOC 2.0 originally launched with a rolling three-month search window. However, an important update in January 2018 extended the historical search capability.

- The API can search up to one year of historical coverage, while its default search window remains three months.

##### 1.2 Maximum number of returned articles

According to the official documentation:

| Setting | Maximum/Default |
|---|---:|
| Default `maxrecords` | 75 |
| Maximum `maxrecords` | 250 |
| Applicable modes | Article List and ImageCollage |

- Even if 1,000 articles match your query, a single Article List request cannot return all 1,000. You need to divide your search into smaller time windows or narrower queries.

A search for NVIDIA does not guarantee that every returned article is entirely about NVIDIA.

For example:
- An article about the AI industry may mention NVIDIA once.
- An article about another technology company may mention NVIDIA in a comparison.
- An article about NVIDIA's financial performance may be highly relevant.

##### 1.3 Search coverage and language

- GDELT monitors news sources globally and supports searching across 65 machine-translated languages.

### API usage rules

##### 2.1 Date and time parameters

- GDELT supports two ways to define your search period:

    Option A – Relative period

    Option B – Specific date range

##### 2.2 Supported response formats

- json , csv , rss , rssarchive , jsonfeed

##### 2.3 Query construction rules

There are some important syntax rules:
- Use quotation marks for exact phrases.
- Use uppercase OR between alternative search terms.
- Boolean OR blocks cannot be nested.
- Put search operators inside the query parameter.
- Use URL encoding when constructing requests manually.

##### 2.4 Request frequency and server load

*One important finding:* the official DOC 2.0 documentation does not specify a fixed public rate limit such as 10 requests per minute.

Therefore, we should not invent a quota.

Nevertheless, I recommend these practices:
1. Avoid sending multiple requests simultaneously.
2. Add a small delay between requests.
3. Use a reasonable timeout.
4. Handle HTTP errors.
5. Save successful results instead of requesting them repeatedly.
6. Retry temporary failures with exponential backoff.

- sooooooo : 

Here is our protocol : 

# 📋 NVIDIA News Collection Protocol

> **Objective:** Establish a consistent data collection methodology for all team members to ensure data quality, reproducibility, and avoid unnecessary API requests.

## 🔧 1. API Configuration

| Setting | Recommended Value |
|---|---|
| API | GDELT DOC 2.0 |
| Mode | `ArtList` |
| Response Format | `json` |
| Maximum Records | `250` |
| Initial Language | English |
| Search Keywords | `NVIDIA`, `NVDA` |
| Sorting | `DateDesc` |
| Request Timeout | 30 seconds |

## ⏱️ 2. Request Management

- Add a **1–2 second delay** between consecutive requests.
- Avoid sending multiple requests simultaneously.
- Set a reasonable timeout for every HTTP request.
- Implement HTTP error handling.
- Use retry mechanisms with exponential backoff for temporary failures.
- Save successful results to avoid repeating identical requests.

## 🧹 3. Data Quality and Cleaning

- Remove duplicate articles using their URLs.
- Check for missing headlines and URLs.
- Verify article relevance to NVIDIA.
- Preserve the original GDELT metadata.
- Keep publication timestamps in a consistent timezone (UTC).
- Distinguish GDELT's `seendate` from the publisher's actual publication date.

## 🌐 4. Web Scraping Rules

Before extracting article content:

- Check the website's `robots.txt`.
- Review the publisher's Terms of Service.
- Check HTTP response status.
- Identify inaccessible pages and paywalls.
- Respect copyright and content usage restrictions.
- Avoid excessive requests to the same website.
- Never attempt to bypass access restrictions.

## 💾 5. Data Storage

**Recommended output format:** CSV (UTF-8)

Suggested filename:

`nvidia_articles_[member]_[start_date]_[end_date].csv`

Recommended columns:

| Column | Description |
|---|---|
| `article_id` | Unique internal article identifier |
| `title` | Original article headline |
| `url` | Original article URL |
| `domain` | Publisher's domain |
| `seendate` | Date/time recorded by GDELT |
| `language` | Article language |
| `sourcecountry` | Source country |
| `collection_date` | Date the data was collected |
| `collection_period` | Assigned collection period |

## 👥 6. Team Collaboration Rules

- Each member must work on an assigned, non-overlapping date range.
- All members must use the same API configuration and collection functions.
- Each member must export their results separately.
- Collection errors and inaccessible URLs must be documented.
- All datasets must be consolidated and deduplicated before sentiment analysis.

---

### ✅ Final Objective

Produce a clean, structured, and reproducible NVIDIA financial news dataset that can be used for headline extraction, sentiment analysis, and integration with historical stock market data.

**Important:** The timeout, delay, and workflow settings above are recommended project practices, not official GDELT rate limits.

# 02. Environment Setup

In [1]:
%pip install requests beautifulsoup4 lxml trafilatura

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Importing required libraries

In [2]:
# API requests
import requests

# Data manipulation
import pandas as pd

# HTML parsing and article extraction
from bs4 import BeautifulSoup
import trafilatura

# Date and time management
from datetime import datetime, timezone

# Request management
import time

# JSON handling
import json

print("Environment successfully initialized!")

c:\Users\PC\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\PC\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Environment successfully initialized!


### Defining configuration variables

In [3]:
# ==========================================
# GDELT DOC 2.0 - CONFIGURATION
# ==========================================

# API endpoint
BASE_URL = "https://api.gdeltproject.org/api/v2/doc/doc"


# API request parameters
MODE = "ArtList"
RESPONSE_FORMAT = "json"
MAX_RECORDS = 250
SORT_ORDER = "DateDesc"


# Request settings
REQUEST_TIMEOUT = 30
REQUEST_DELAY = 5

print("GDELT configuration initialized successfully.")

GDELT configuration initialized successfully.


# 03. Building Our First API Request

### Defining NVIDIA search queries

In [4]:
# ==========================================
# NVIDIA SEARCH QUERY BANK
# ==========================================

SEARCH_QUERIES = {
    "company_identity": [
        "NVIDIA",
        '"NVIDIA Corporation"'
    ],

    "stock_market": [
        "NVDA stock",
        "NVDA shares",
        "NVIDIA stock price",
        "NVIDIA shares"
    ],

    "financial_results": [
        "NVIDIA earnings",
        "NVIDIA revenue",
        "NVIDIA EPS",
        "NVIDIA financial results",
        "NVIDIA earnings guidance"
    ],

    "analyst_sentiment": [
        "NVIDIA analyst rating",
        "NVIDIA price target",
        "NVIDIA stock upgrade",
        "NVIDIA stock downgrade"
    ],

    "products_technology": [
        "NVIDIA GPU",
        "NVIDIA Blackwell",
        "NVIDIA Vera Rubin",
        "NVIDIA CUDA",
        "NVIDIA AI chips"
    ],

    "ai_datacenters": [
        "NVIDIA AI infrastructure",
        "NVIDIA data center",
        "NVIDIA cloud computing",
        "NVIDIA AI computing"
    ],

    "partnerships": [
        "NVIDIA partnership",
        "NVIDIA Microsoft",
        "NVIDIA Amazon",
        "NVIDIA Google",
        "NVIDIA Oracle"
    ],

    "competition": [
        "NVIDIA AMD",
        "NVIDIA Intel",
        "NVIDIA AI chip competition"
    ],

    "regulation_geopolitics": [
        "NVIDIA export controls",
        "NVIDIA China",
        "NVIDIA semiconductor restrictions",
        "NVIDIA tariffs"
    ],

    "supply_chain": [
        "NVIDIA TSMC",
        "NVIDIA semiconductor supply",
        "NVIDIA HBM memory",
        "NVIDIA chip manufacturing"
    ],

    "management": [
        '"Jensen Huang" NVIDIA',
        "NVIDIA CEO announcement"
    ],

    "corporate_actions": [
        "NVIDIA dividend",
        "NVIDIA share buyback",
        "NVIDIA acquisition",
        "NVIDIA stock split"
    ]
}

# Display all queries
for category, queries in SEARCH_QUERIES.items():
    print(f"\n{category.upper()}")

    for query in queries:
        print(f"  - {query}")


COMPANY_IDENTITY
  - NVIDIA
  - "NVIDIA Corporation"

STOCK_MARKET
  - NVDA stock
  - NVDA shares
  - NVIDIA stock price
  - NVIDIA shares

FINANCIAL_RESULTS
  - NVIDIA earnings
  - NVIDIA revenue
  - NVIDIA EPS
  - NVIDIA financial results
  - NVIDIA earnings guidance

ANALYST_SENTIMENT
  - NVIDIA analyst rating
  - NVIDIA price target
  - NVIDIA stock upgrade
  - NVIDIA stock downgrade

PRODUCTS_TECHNOLOGY
  - NVIDIA GPU
  - NVIDIA Blackwell
  - NVIDIA Vera Rubin
  - NVIDIA CUDA
  - NVIDIA AI chips

AI_DATACENTERS
  - NVIDIA AI infrastructure
  - NVIDIA data center
  - NVIDIA cloud computing
  - NVIDIA AI computing

PARTNERSHIPS
  - NVIDIA partnership
  - NVIDIA Microsoft
  - NVIDIA Amazon
  - NVIDIA Google
  - NVIDIA Oracle

COMPETITION
  - NVIDIA AMD
  - NVIDIA Intel
  - NVIDIA AI chip competition

REGULATION_GEOPOLITICS
  - NVIDIA export controls
  - NVIDIA China
  - NVIDIA semiconductor restrictions
  - NVIDIA tariffs

SUPPLY_CHAIN
  - NVIDIA TSMC
  - NVIDIA semiconductor supply


## 03.2. Search Query Evaluation Strategy

### 🎯 Objective

The objective of this evaluation is to identify the most effective search queries for collecting NVIDIA-related financial news through the GDELT DOC 2.0 API.

Rather than selecting queries based only on their keywords, we will evaluate their ability to retrieve relevant, diverse, and accessible articles.

Each candidate query will be tested independently using the same API configuration and collection period to ensure a fair comparison.

### 📊 1. Evaluation Metrics

| Metric | Description | Purpose |
|---|---|---|
| Retrieved Articles | Total number of articles returned by GDELT for a query | Measure search coverage |
| Relevant Articles | Number of retrieved articles genuinely related to NVIDIA | Measure relevance |
| Irrelevant Articles | Number of articles unrelated to NVIDIA or the project's research objective | Identify search noise |
| Duplicate URLs | Number of URLs already retrieved by other queries | Measure overlap |
| Unique Articles | Number of new articles contributed by a query | Measure additional coverage |
| Extraction Success | Number of accessible articles from which content or headlines can be extracted | Evaluate scraping feasibility |

### 📐 2. Evaluation Indicators

We will use the following indicators to compare the performance of candidate queries.

**A. Query Precision**

Measures the proportion of retrieved articles that are relevant.

$$
Precision = \frac{Relevant\ Articles}{Retrieved\ Articles}
$$

**B. Irrelevant Article Rate**

Measures the proportion of retrieved articles that do not match our research objective.

$$
Irrelevant\ Rate = \frac{Irrelevant\ Articles}{Retrieved\ Articles}
$$

**C. Unique Contribution**

Measures how many relevant articles a query adds to the dataset that have not already been collected by other queries.

$$
Unique\ Contribution = \frac{New\ Relevant\ Articles}{All\ Relevant\ Articles\ Retrieved}
$$

**D. Extraction Success Rate**

Measures the proportion of tested article URLs from which the required content can be extracted successfully.

$$
Extraction\ Success = \frac{Successfully\ Extracted\ Articles}{Tested\ Article\ URLs}
$$

*Note: Metrics are calculated only when their denominators are greater than zero. Unique contribution should be evaluated after URL deduplication.*

### 🔬 3. Experimental Procedure

To ensure a consistent evaluation, we will follow these steps:

1. Select a candidate query from the search query bank.
2. Send an API request using the same collection period, response format, sorting order, and maximum record setting.
3. Record the number of articles returned by GDELT.
4. Inspect the retrieved headlines and classify articles as relevant or irrelevant.
5. Compare article URLs against previously collected results to identify duplicates.
6. Record the number of unique relevant articles contributed by the query.
7. Select a small sample of URLs and test whether their headlines or article content can be extracted.
8. Store the evaluation results in a comparison table.

### 📋 4. Query Evaluation Results

The following table will be populated after testing the candidate queries.

| Query ID | Category | Retrieved | Relevant | Irrelevant | Duplicates | Unique Relevant | Extraction Success |
|---|---|---:|---:|---:|---:|---:|---:|
| Q01 | Company Identity | | | | | | |
| Q02 | Stock Market | | | | | | |
| Q03 | Financial Results | | | | | | |
| Q04 | Analyst Sentiment | | | | | | |
| Q05 | Products & Technology | | | | | | |
| Q06 | AI & Data Centers | | | | | | |
| Q07 | Partnerships | | | | | | |
| Q08 | Competition | | | | | | |
| Q09 | Regulation & Geopolitics | | | | | | |
| Q10 | Supply Chain | | | | | | |
| Q11 | Management | | | | | | |
| Q12 | Corporate Actions | | | | | | |

### 🏆 5. Query Selection Criteria

After evaluating all candidate queries, we will prioritize queries that:

- Retrieve a high proportion of relevant NVIDIA-related articles.
- Contribute new articles rather than repeatedly retrieving the same URLs.
- Cover different financial and business dimensions of NVIDIA.
- Provide accessible article URLs suitable for headline or content extraction.
- Avoid excessive irrelevant results caused by ambiguous keywords.

A query should not be selected solely because it retrieves the largest number of articles. Relevance and unique contribution are equally important.

### ⚠️ 6. Evaluation Limitations

- GDELT's returned article count does not necessarily represent all articles published during the selected period.
- The maximum number of records per request may limit the observed coverage.
- Search results can overlap significantly across different queries.
- Article accessibility depends on the publisher's website, paywalls, and technical restrictions.
- GDELT's `seendate` represents when GDELT observed the article, which may differ from its original publication date.
- Manual relevance classification may introduce subjective decisions; therefore, consistent classification rules should be applied.

### ✅ 7. Expected Outcome

At the end of this evaluation, we aim to obtain:

1. A selected set of reliable NVIDIA financial news queries.
2. A comparison of their retrieval performance and relevance.
3. An estimate of duplicate and unique article contributions.
4. An initial assessment of article URL accessibility.
5. A standardized search strategy that can be shared and reproduced by all team members.

**Final objective:** Build a relevant, diverse, and reproducible financial news dataset that can later be integrated with NVIDIA historical stock market data for sentiment analysis and next-day stock direction prediction.


### Selecting the date range

- Initial query-testing period : 26 Oct 2026 to 1 Nov 2026

In [5]:
# ==========================================
# DATE RANGE CONFIGURATION
# ==========================================

START_DATE = "20261026000000"
END_DATE = "20261101235959"

TIME_ZONE = "UTC"

print("Collection period:")
print(f"Start: {START_DATE}")
print(f"End:   {END_DATE}")

Collection period:
Start: 20261026000000
End:   20261101235959


### Setting request parameters

In [6]:
# ==========================================
# GDELT API REQUEST PARAMETERS
# ==========================================

params = {
    "mode": MODE,
    "format": RESPONSE_FORMAT,
    "maxrecords": MAX_RECORDS,
    "sort": SORT_ORDER,
    "startdatetime": START_DATE,
    "enddatetime": END_DATE
}

print("Request parameters successfully configured.")

for key, value in params.items():
    print(f"{key}: {value}")

Request parameters successfully configured.
mode: ArtList
format: json
maxrecords: 250
sort: DateDesc
startdatetime: 20261026000000
enddatetime: 20261101235959


### Sending GET requests

In [7]:
import time
import requests

MIN_INTERVAL = 6.0          # GDELT exige >= 5 s entre deux requêtes ; marge de sécurité
BACKOFF_BASE = 15           # secondes : 15, 30, 60...
MAX_WAIT = 120
REQUEST_TIMEOUT = 60        # GDELT met souvent 15-25 s à répondre

session = requests.Session()
session.headers.update({"User-Agent": "Mozilla/5.0 (research script)"})

_last_request_time = 0.0
_cache = {}                 # évite de redemander une requête déjà faite


def _throttle():
    """Garantit MIN_INTERVAL secondes entre deux requêtes réelles (retries inclus)."""
    global _last_request_time
    elapsed = time.monotonic() - _last_request_time
    if elapsed < MIN_INTERVAL:
        time.sleep(MIN_INTERVAL - elapsed)
    _last_request_time = time.monotonic()


def fetch_gdelt_articles(query, max_retries=5):
    if query in _cache:
        return _cache[query]

    request_params = {**params, "query": query}

    for attempt in range(max_retries):
        _throttle()
        wait = min(BACKOFF_BASE * (2 ** attempt), MAX_WAIT)

        try:
            response = session.get(BASE_URL, params=request_params, timeout=REQUEST_TIMEOUT)
        except (requests.exceptions.Timeout, requests.exceptions.ConnectionError) as error:
            print(f"Network error ({error}). Retry in {wait}s...")
            time.sleep(wait)
            continue

        # Limitation de débit (HTTP 429, ou message texte avec statut 200)
        body = response.text.strip()
        if response.status_code == 429 or "limit requests" in body.lower():
            print(f"Rate limited. Waiting {wait}s...")
            time.sleep(wait)
            continue

        # Erreurs serveur : on réessaie
        if response.status_code >= 500:
            print(f"Server error {response.status_code}. Retry in {wait}s...")
            time.sleep(wait)
            continue

        # Erreur client (autre que 429) : inutile de réessayer
        if not response.ok:
            print(f"HTTP {response.status_code}: {body[:200]}")
            return None

        try:
            data = response.json()
        except ValueError:
            # GDELT renvoie du texte brut pour une requête invalide
            # (ex. mot < 3 caractères, parenthèse manquante...)
            print(f"Non-JSON response for '{query}': {body[:200]}")
            return None

        _cache[query] = data
        return data

    print(f"Gave up on '{query}' after {max_retries} attempts.")
    return None

##### Test 

In [ ]:
import requests

r = requests.get(
    "https://api.gdeltproject.org/api/v2/doc/doc",
    params={"query": "NVIDIA", "mode": "artlist", "format": "json", "maxrecords": 5, "timespan": "1d"},
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=60,
)
print(r.status_code)
print(dict(r.headers))
print(r.text[:500])

In [ ]:
query_results = {}

for category, queries in SEARCH_QUERIES.items():
    for query in queries:
        print(f"Searching: {query}")
        result = fetch_gdelt_articles(query)
        if result is not None:
            query_results[query] = result

print(f"\nCompleted: {len(query_results)} queries returned valid JSON.")

Searching: NVIDIA
Rate limited. Waiting 15s...
Rate limited. Waiting 30s...
Rate limited. Waiting 60s...
Rate limited. Waiting 120s...


### Checking HTTP responses

# 04. Exploring the API Response

### Understanding JSON structure

### Extracting article metadata

### Converting JSON into a Pandas DataFrame

### Inspecting the first results

# 05. Data Cleaning and Export

### Selecting relevant columns

### Handling missing values

### Removing duplicate URLs

### Exporting the CSV file

# 06. Article URL Exploration and Headline Extraction

### Selecting sample URLs

### Inspecting HTML page structure

### Extracting headlines using BeautifulSoup

### Comparing extracted headlines with GDELT titles

### Handling inaccessible pages

# 07. Results Evaluation

### Number of retrieved articles

### Percentage of accessible URLs

### Successful headline extraction rate

### Main difficulties and limitations

# 08. Lessons Learned and Team Collection Strategy

### Recommended collection procedure

### Rules for avoiding duplicate work

### Date allocation strategy

### Recommendations for the final team notebook